In [ ]:
# Build the class dataset: the full Parquet (benign + attack) converted to one CSV.
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("build_class_dataset")
         .config("spark.driver.memory", "8g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

base = ("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/correlated_dataset/"
        "2026-09-26_00-28-45/finalized_data")
out  = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"

# Column order taken from the sample CSV header, so the file matches what the notebook expects
cols = spark.read.option("header", True).csv(base + "/Sample_Dataset_CSV/Benign/part-*.csv").columns

full = spark.read.parquet(base + "/Full_Dataset_Parquet/2026-*")
full = full.withColumn("datetime", F.date_format("datetime", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
full = full.select([(F.col(c).cast("string") if c in full.columns
                     else F.lit(None).cast("string")).alias(c) for c in cols])

print("Total rows:", full.count())
full.groupBy("label_tactic").count().orderBy("count", ascending=False).show(truncate=False)

full.coalesce(1).write.mode("overwrite").option("header", True).csv(out)
print("Written to", out)

In [ ]:
# pull attack output to compare to mission logs
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "8g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

df = (spark.read.option("header", True)
      .csv("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"))
summary = (df.filter((F.col("label_tactic") != "none") & (F.col("label_technique") != "Duplicate"))
             .withColumn("day", F.substring("datetime", 1, 10))
             .groupBy("src_ip_zeek", "dest_ip_zeek", "label_technique", "label_tactic", "day")
             .agg(F.count(F.lit(1)).alias("connections"),
                  F.countDistinct("dest_port_zeek").alias("distinct_dest_ports"),
                  F.min("datetime").alias("first"), F.max("datetime").alias("last"))
             .orderBy("first"))
pdf = summary.toPandas()
pdf.to_csv("/home/ubuntu/CNT4416_attack_summary.csv", index=False)
print(len(pdf), "rows")
#